# SigAlg's `L2.norm` method

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2.norm` method in SigAlg computes the *$L^2$-norm* of a random variable in an $L^2$-space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2.norm).

## Mathematical definition

Let $X \in L^2(\Omega, \mathcal{F}, P)$ be a square-integrable random variable on a probability space $(\Omega, \mathcal{F}, P)$. The *$L^2$-norm* of $X$ is defined as

$$
\|X\| \stackrel{\text{def}}{=} \sqrt{\langle X, X \rangle} = \sqrt{E(X^2)} = \sqrt{\int_\Omega X^2 \, dP}.
$$

The $L^2$-norm satisfies the following properties:

1. *Positivity*: For all $X \in L^2(\Omega, \mathcal{F}, P)$,
   $$
   \|X\| \geq 0,
   $$
   with equality if and only if $X = 0$ almost surely.

2. *Homogeneity*: For all $X \in L^2(\Omega, \mathcal{F}, P)$ and $a \in \mathbb{R}$,
   $$
   \|aX\| = |a| \cdot \|X\|.
   $$

3. *Triangle inequality*: For all $X, Y \in L^2(\Omega, \mathcal{F}, P)$,
   $$
   \|X + Y\| \leq \|X\| + \|Y\|.
   $$

## API examples


### Basic norms

We begin by setting up a probability space and creating an $L^2$-space.

In [2]:
from sigalg.core import ProbabilityMeasure, RandomVariable, SampleSpace, SigmaAlgebra
from sigalg.l2 import L2

Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.45,
        3: 0.3,
    }
)

H = L2(sample_space=Omega, sig_alg=F, prob_measure=P)

Create a random variable and compute its $L^2$-norm.

In [3]:
X = RandomVariable(domain=Omega, name="X").from_dict(
    {
        0: 2,
        1: -3,
        2: 2,
        3: -3,
    }
).with_probability_measure(P)

norm_X = H.norm(X)
print(X)
print(f"||X|| = {norm_X:.2f}")

Random variable 'X':
        X
sample   
0       2
1      -3
2       2
3      -3
||X|| = 2.50


We can verify that the norm equals $\sqrt{E(X^2)}$ using the `expectation` method.

In [4]:
import numpy as np

from sigalg.core import Operators

E = Operators.expectation

E_X2 = E(X**2).item()
print(f"sqrt(E(X^2)) = {np.sqrt(E_X2):.2f}")

sqrt(E(X^2)) = 2.50


### Homogeneity

The norm is homogeneous: $\|aX\| = |a| \cdot \|X\|$ for any scalar $a$.

In [5]:
a = -2.5

norm_aX = H.norm(a * X)
abs_a_times_norm_X = abs(a) * H.norm(X)

print(f"||{a}X|| = {norm_aX:.2f}")
print(f"|{a}| · ||X|| = {abs_a_times_norm_X:.2f}")

||-2.5X|| = 6.25
|-2.5| · ||X|| = 6.25


### Triangle inequality

The norm satisfies the triangle inequality: $\|X + Y\| \leq \|X\| + \|Y\|$.

In [6]:
Y = (
    RandomVariable(domain=Omega, name="Y")
    .from_dict(
        {
            0: 1,
            1: 4,
            2: 1,
            3: 4,
        }
    )
    .with_probability_measure(P)
)

norm_X_plus_Y = H.norm(X + Y)
norm_X_plus_norm_Y = H.norm(X) + H.norm(Y)

print(f"||X + Y|| = {norm_X_plus_Y:.2f}")
print(f"||X|| + ||Y|| = {norm_X_plus_norm_Y:.2f}")

||X + Y|| = 2.32
||X|| + ||Y|| = 5.28


### Connection to variance and standard deviation

The *uncentered norm* $\|X\|$ measures the overall magnitude of $X$, while the *centered norm* $\|X - E(X)\|$ measures the variability around the mean. Indeed, we have

$$
\| X - E(X) \| = \sqrt{\langle X - E(X), X - E(X) \rangle} = \sqrt{E((X - E(X))^2)} = \sqrt{V(X)} = \sigma(X),
$$

where $V(X)$ and $\sigma(X)$ are the variance and standard deviation of $X$, respectively. We can test these equalities using SigAlg's `var` and `std` methods.

In [7]:
from sigalg.core import Operators

E = Operators.expectation
V = Operators.variance
std = Operators.std

centered_norm = H.norm(X - E(X))
variance_X = V(X).item()
std_X = std(X).item()

print(f"||X - E(X)|| = {centered_norm:.4f}")
print(f"sqrt(V(X)) = {np.sqrt(variance_X):.4f}")
print(f"std(X) = {std_X:.4f}")

||X - E(X)|| = 2.4875
sqrt(V(X)) = 2.4875
std(X) = 2.4875
